In [13]:
import ast
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

BLOCK_SIZE = 64
ANNOTATIONS_PATH = "../data/annotations.csv"
EMBEDDINGS_DIR = Path("../jepa2/embeddings")

In [14]:
# Load only relevant annotations
df = pd.read_csv(ANNOTATIONS_PATH)
df = df[df["relevant"] == True].copy()

# Convert frame ranges to block ranges
df["start_block"] = df["start_frame"] // BLOCK_SIZE
df["stop_block"] = df["stop_frame"] // BLOCK_SIZE

# Expand each action to one row per block it spans
rows = []
for _, row in df.iterrows():
    # all_noun_classes is a list like [3] or [1, 2, 5] — an action can have multiple nouns.
    # We use the first one as the primary noun class, which corresponds to the primary noun.
    # If a block has multiple overlapping actions, we later resolve the conflict by majority vote.
    primary_noun_class = ast.literal_eval(row["all_noun_classes"])[0]
    for block in range(row["start_block"], row["stop_block"] + 1):
        rows.append({
            "video_id": row["video_id"],
            "block": block,
            "verb_class": row["verb_class"],
            "noun_class": primary_noun_class,
        })

expanded = pd.DataFrame(rows)

# If multiple actions overlap the same block, pick the most frequent label
verb_labels = expanded.groupby(["video_id", "block"])["verb_class"].agg(lambda x: x.mode()[0])
noun_labels = expanded.groupby(["video_id", "block"])["noun_class"].agg(lambda x: x.mode()[0])

print(f"Relevant blocks: {len(verb_labels)}")
print(f"Unique verb classes: {verb_labels.nunique()}  Unique noun classes: {noun_labels.nunique()}")

Relevant blocks: 142433
Unique verb classes: 59  Unique noun classes: 271


In [15]:
# Load embeddings only for blocks that have a label
all_embeddings = []
all_verb_labels = []
all_noun_labels = []

for pkl_path in sorted(EMBEDDINGS_DIR.glob("*.pkl")):
    video_id = pkl_path.stem

    with pkl_path.open("rb") as f:
        payload = pickle.load(f)

    embeddings = payload["embeddings"] if isinstance(payload, dict) else payload

    for block_idx in range(len(embeddings)):
        key = (video_id, block_idx)
        if key not in verb_labels.index:
            continue  # skip irrelevant blocks
        all_embeddings.append(embeddings[block_idx])
        all_verb_labels.append(verb_labels[key])
        all_noun_labels.append(noun_labels[key])

X = torch.tensor(np.stack(all_embeddings), dtype=torch.float32)

# Encode labels as contiguous integers starting from 0
verb_encoder = LabelEncoder().fit(all_verb_labels)
noun_encoder = LabelEncoder().fit(all_noun_labels)

y_verb = torch.tensor(verb_encoder.transform(all_verb_labels), dtype=torch.long)
y_noun = torch.tensor(noun_encoder.transform(all_noun_labels), dtype=torch.long)

n_verb_classes = len(verb_encoder.classes_)
n_noun_classes = len(noun_encoder.classes_)

print(f"Loaded {len(X)} relevant blocks")
print(f"Verb classes: {n_verb_classes}  Noun classes: {n_noun_classes}")

Loaded 5042 relevant blocks
Verb classes: 36  Noun classes: 120


In [16]:
def make_classifier(input_dim, num_classes):
    return nn.Sequential(
        nn.Linear(input_dim, 512),
        nn.LayerNorm(512),
        nn.GELU(),
        nn.Dropout(0.25),
        nn.Linear(512, 256),
        nn.LayerNorm(256),
        nn.GELU(),
        nn.Dropout(0.25),
        nn.Linear(256, 128),
        nn.LayerNorm(128),
        nn.GELU(),
        nn.Dropout(0.25),
        nn.Linear(128, num_classes),
    )

def train_model(model, X_train, y_train, device, epochs=30, batch_size=256, lr=1e-3):
    model = model.to(device)
    loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(y_batch)
        print(f"epoch={epoch:02d}  loss={total_loss / len(y_train):.4f}")
    return model

def evaluate_model(model, X_test, y_test, device):
    model.eval()
    with torch.no_grad():
        logits = model(X_test.to(device)).cpu()
        preds = logits.argmax(dim=-1)
    acc = (preds == y_test).float().mean().item()
    print(f"accuracy: {acc:.3f}  ({int(acc * len(y_test))}/{len(y_test)} correct)")

device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
input_dim = X.shape[1]
print(f"Device: {device}  Input dim: {input_dim}")

Device: mps  Input dim: 1024


In [17]:
# --- Verb classifier ---
# Drop classes with fewer than 2 samples — stratified split requires at least 2 per class
verb_counts = y_verb.bincount()
valid_verb_mask = verb_counts[y_verb] >= 2
X_verb, y_verb_filtered = X[valid_verb_mask], y_verb[valid_verb_mask]
print(f"Dropped {(~valid_verb_mask).sum().item()} blocks with singleton verb classes")

X_train, X_test, y_train, y_test = train_test_split(X_verb, y_verb_filtered, test_size=0.2, random_state=42, stratify=y_verb_filtered)

print("Training verb classifier...")
verb_model = train_model(make_classifier(input_dim, n_verb_classes), X_train, y_train, device)

print("\nVerb classifier results:")
evaluate_model(verb_model, X_test, y_test, device)

Dropped 1 blocks with singleton verb classes
Training verb classifier...
epoch=01  loss=2.5300
epoch=02  loss=1.9013
epoch=03  loss=1.6598
epoch=04  loss=1.4993
epoch=05  loss=1.3546
epoch=06  loss=1.2375
epoch=07  loss=1.1338
epoch=08  loss=1.0571
epoch=09  loss=0.9486
epoch=10  loss=0.8927
epoch=11  loss=0.8318
epoch=12  loss=0.7753
epoch=13  loss=0.7129
epoch=14  loss=0.6636
epoch=15  loss=0.6106
epoch=16  loss=0.5650
epoch=17  loss=0.5511
epoch=18  loss=0.4897
epoch=19  loss=0.4294
epoch=20  loss=0.4124
epoch=21  loss=0.4258
epoch=22  loss=0.4025
epoch=23  loss=0.3508
epoch=24  loss=0.3337
epoch=25  loss=0.3166
epoch=26  loss=0.2790
epoch=27  loss=0.2894
epoch=28  loss=0.2434
epoch=29  loss=0.2568
epoch=30  loss=0.2352

Verb classifier results:
accuracy: 0.731  (737/1009 correct)


In [7]:
# --- Noun classifier ---
# Drop classes with fewer than 2 samples — stratified split requires at least 2 per class
noun_counts = y_noun.bincount()
valid_noun_mask = noun_counts[y_noun] >= 2
X_noun, y_noun_filtered = X[valid_noun_mask], y_noun[valid_noun_mask]
print(f"Dropped {(~valid_noun_mask).sum().item()} blocks with singleton noun classes")

X_train, X_test, y_train, y_test = train_test_split(X_noun, y_noun_filtered, test_size=0.2, random_state=42, stratify=y_noun_filtered)

print("Training noun classifier...")
noun_model = train_model(make_classifier(input_dim, n_noun_classes), X_train, y_train, device)

print("\nNoun classifier results:")
evaluate_model(noun_model, X_test, y_test, device)

Dropped 3 blocks with singleton noun classes
Training noun classifier...
epoch=01  loss=4.2990
epoch=02  loss=3.7002
epoch=03  loss=3.2818
epoch=04  loss=2.8934
epoch=05  loss=2.6019
epoch=06  loss=2.3134
epoch=07  loss=2.0889
epoch=08  loss=1.8719
epoch=09  loss=1.7013
epoch=10  loss=1.5662
epoch=11  loss=1.3977
epoch=12  loss=1.2924
epoch=13  loss=1.1482
epoch=14  loss=1.0486
epoch=15  loss=0.9715
epoch=16  loss=0.8874
epoch=17  loss=0.8187
epoch=18  loss=0.7516
epoch=19  loss=0.6877
epoch=20  loss=0.6256
epoch=21  loss=0.5911
epoch=22  loss=0.5389
epoch=23  loss=0.5040
epoch=24  loss=0.4435
epoch=25  loss=0.4444
epoch=26  loss=0.4037
epoch=27  loss=0.3819
epoch=28  loss=0.3660
epoch=29  loss=0.3435
epoch=30  loss=0.3107

Noun classifier results:
accuracy: 0.718  (724/1008 correct)
